### Causal Inference

Methods for estimating a treatment's real causal effect from data that wasn't generated by a clean randomized experiment. `ab-testing.ipynb` covers the RCT case, where randomization itself removes confounding; this notebook covers what to do when you can't randomize (backtests, historical rollouts, natural experiments) and confounding is a live threat. Covers confounding, Simpson's paradox, propensity score matching vs. inverse propensity weighting, difference-in-differences, synthetic control, and instrumental variables / regression discontinuity as brief quasi-experimental alternatives.

## Confounding and Simpson's Paradox

#### 0. Core idea

A confounder is a variable that affects both treatment assignment (or, in a purely observational setting, which group a unit ends up in) and the outcome, creating a spurious association between two things with no direct causal link at all between them. The textbook version needs no reversal: ice cream sales and drowning deaths correlate strongly across months, but neither causes the other, both are driven by hot weather, which increases both ice cream purchases and swimming (and therefore drowning risk).

Simpson's paradox is the more dramatic special case: a trend visible in every subgroup reverses (or vanishes) once the subgroups are combined into an aggregate. Classic example, kidney stone treatment (stone size is the confounder: doctors preferentially give the more invasive Treatment A to harder, large-stone cases):

```
                Small stones          Large stones          Overall
Treatment A     81/87  = 93.1%        192/263 = 73.0%        273/350 = 78.0%
Treatment B     234/270 = 86.7%       55/80   = 68.8%        289/350 = 82.6%
```

Treatment A wins in BOTH subgroups individually, but Treatment B wins overall. The reversal happens because the case MIX differs: A is used mostly on the harder large-stone cases (263 of its 350 cases, 75%), while B is used mostly on the easier small-stone cases (270 of its 350 cases, 77%). B's overall average looks better only because it was given an easier caseload, not because it's actually more effective. Never trust an aggregate comparison between two groups without checking whether the *mix* of confounders (case difficulty, cohort, time period) differs between them.

In [ ]:
small_A_succ, small_A_n = 81, 87
small_B_succ, small_B_n = 234, 270
large_A_succ, large_A_n = 192, 263
large_B_succ, large_B_n = 55, 80

print("small stones  -- A:", round(small_A_succ/small_A_n, 3), " B:", round(small_B_succ/small_B_n, 3))
print("large stones  -- A:", round(large_A_succ/large_A_n, 3), " B:", round(large_B_succ/large_B_n, 3))

overall_A = (small_A_succ + large_A_succ) / (small_A_n + large_A_n)
overall_B = (small_B_succ + large_B_succ) / (small_B_n + large_B_n)
print("overall       -- A:", round(overall_A, 3), " B:", round(overall_B, 3))

# the confound: A is used on a much harder case mix (share of large-stone cases)
print("share of large-stone cases -- A:", round(large_A_n/(small_A_n+large_A_n), 3),
      " B:", round(large_B_n/(small_B_n+large_B_n), 3))

#### 1. Detecting it before it burns you

Whenever an aggregate trend is surprising, or contradicts a narrative you'd otherwise expect, deliberately slice the same comparison by any plausible confounding dimension (cohort, region, time period, case-difficulty tier, a self-selection flag like opted-in-vs-not) before trusting the aggregate number at all. If the direction of the effect flips or vanishes within slices the way it does above, that is the paradox, and it means the aggregate comparison is confounded by an uneven distribution across that dimension rather than reflecting one uniform, real effect. The same trap shows up in ordinary product metrics without needing a reversal: users who engage with a new opt-in feature retain better than users who don't, but if only already-loyal power users chose to opt in, the correlation reflects self-selection into usage, not the feature's real causal effect, exactly why a randomized rollout, not an observational opted-in-vs-not comparison, is what makes a real causal claim possible.

## Propensity Score Matching vs. Inverse Propensity Weighting (IPW)

#### 0. Core idea

Both correct for a *known, measured* confounder Z that affects both treatment assignment and outcome. The propensity score e(Z) = P(T=1 | Z) is the probability of being treated given the confounder.

- **PSM**: match each treated unit to an untreated unit with a similar propensity score, then compare outcomes only within matched pairs. Throws away unmatched units.
- **IPW**: keep every unit, but re-weight it by 1/e(Z) if treated or 1/(1-e(Z)) if untreated, so that after weighting, treated and untreated groups look balanced on Z. Uses all the data, but is more sensitive to extreme propensity scores near 0 or 1 (huge weights).

Toy setup, confounder Z in {0, 1} (e.g. young / old). Old units are both more likely to be treated AND have a higher baseline outcome, so the confound biases a naive comparison:

```
Z=0 (young): 3 untreated (Y=0 each), 1 treated (Y=5)   -> P(T=1|Z=0) = 1/4 = 0.25
Z=1 (old):   1 untreated (Y=10),     3 treated (Y=15 each) -> P(T=1|Z=1) = 3/4 = 0.75
```
Y is generated as `Y = 10*Z + 5*T` by construction, so the TRUE treatment effect is exactly 5. A naive mean-difference will overstate it because treatment is confounded with Z.

In [ ]:
import numpy as np

Z = np.array([0,0,0,0,1,1,1,1])
T = np.array([0,0,0,1,0,1,1,1])
Y = 10*Z + 5*T  # true effect of T is 5, by construction

naive_ate = Y[T==1].mean() - Y[T==0].mean()
print("naive difference in means:", naive_ate, " (biased, confounded with Z)")

# propensity score is just P(T=1|Z) per stratum here (binary Z makes this exact, no model needed)
e = np.where(Z == 0, 0.25, 0.75)

ipw_terms = T*Y/e - (1-T)*Y/(1-e)
print("IPW terms per unit:", ipw_terms)
print("IPW-adjusted ATE:", ipw_terms.mean(), " (recovers the true effect of 5)")

#### 1. When to use which

PSM is easier to sanity-check by eye (you can literally look at matched pairs) and degrades gracefully when propensity scores are poorly estimated, but throws away any unit it can't match, shrinking your effective sample. IPW uses every observation and is the more common choice when you have a large enough sample and reasonably well-behaved propensity scores (no scores near 0 or 1), but a single badly-mis-estimated extreme weight can dominate the whole estimate. Both only correct for confounders you actually measured and included in Z, neither helps with an unmeasured confounder, that's what the quasi-experimental designs below are for.

IPW has a second, distinct use beyond the treatment-effect setting above: correcting EXPOSURE bias, not just treatment-assignment bias. A ranking or CTR model trained only on logged (item, click/no-click) pairs only ever observes outcomes for items it (or an earlier version of itself) already chose to show, items that were never shown have no label at all, a Missing-Not-At-Random pattern that silently reinforces whatever bias already existed in what got shown. The fix is the identical IPW mechanic: reweight each logged example by the inverse of the probability that item was actually shown (its exposure propensity), which up-weights rarely-shown items back toward what a fully, randomly-exposed dataset would look like. This is why epsilon-greedy exploration (`reinforcement-learning-bandits.ipynb`) is often paired with IPW in production ranking systems, propensity weighting needs non-zero exposure probability everywhere to even be well-defined, and pure exploitation (always showing the current top pick) drives some items' exposure probability to zero.

## Difference-in-Differences (DiD)

#### 0. Core idea

Used when treatment and control groups both existed before and after a treatment event, and you're worried about an unmeasured confounder that's constant within each group but shifts over time for everyone (a general trend, e.g. a firm-wide seasonal effect that would otherwise get mistaken for the treatment's own impact). DiD nets out that common trend: `DiD = (treated_after - treated_before) - (control_after - control_before)`.

Toy 2x2 table:
```
                Before   After   Change
Treatment group   50       70      +20
Control group      40       50      +10
DiD = 20 - 10 = 10   (the part of the treatment group's change NOT explained by the shared trend)
```
Key assumption: absent treatment, the two groups would have moved in *parallel* (the "parallel trends" assumption). If a good comparison group genuinely doesn't exist (only one treated unit, no untreated unit that looks remotely similar), synthetic control, covered right after this section, is the standard fallback.

In [ ]:
import numpy as np

# unit-level data whose group means match the 2x2 table above
treat = np.array([1,1,1,1,0,0,0,0])
post  = np.array([0,0,1,1,0,0,1,1])
Y     = np.array([48,52,68,72,38,42,48,52], dtype=float)
inter = treat * post

t_before, t_after = Y[(treat==1)&(post==0)].mean(), Y[(treat==1)&(post==1)].mean()
c_before, c_after = Y[(treat==0)&(post==0)].mean(), Y[(treat==0)&(post==1)].mean()
did_by_hand = (t_after - t_before) - (c_after - c_before)
print("group means -- treat before/after:", t_before, t_after, " control before/after:", c_before, c_after)
print("DiD by hand:", did_by_hand)

# same estimate falls out as the interaction coefficient in Y ~ treat + post + treat:post
X = np.column_stack([np.ones(len(Y)), treat, post, inter])
coef, *_ = np.linalg.lstsq(X, Y, rcond=None)
print("regression coefficients [const, treat, post, treat:post]:", coef)
print("interaction coefficient == DiD estimate:", round(coef[3], 4))

#### 1. Testing the parallel-trends assumption, not just assuming it

Parallel trends is untestable in the post-period (you never observe the treated group's untreated counterfactual), but it's directly checkable in the PRE-period: plot both groups' outcome trends before treatment even started, if they were already diverging, the DiD estimate is unreliable regardless of what the post-period number says. More formally, this is a placebo test, run the DiD regression restricted to pre-period data only, picking a fake "placebo cutover" point partway through, and check whether the resulting interaction term is indistinguishable from zero. A significant nonzero effect in a window where nothing actually happened yet is a direct violation of the assumption the real post-treatment DiD estimate depends on.

In [ ]:
# placebo test: split the PRE-period only, at a fake cutover point, and check the DiD-style estimate

# case 1: genuinely parallel pre-trends (both moving +5 per period before treatment)
treated_pre_ok = [50.0, 55.0]
control_pre_ok = [40.0, 45.0]
placebo_did_ok = (treated_pre_ok[1]-treated_pre_ok[0]) - (control_pre_ok[1]-control_pre_ok[0])
print("placebo DiD, genuinely parallel pre-trends:", placebo_did_ok, " (near zero, assumption holds)")

# case 2: treated group was ALREADY diverging before treatment (a real violation)
treated_pre_bad = [50.0, 58.0]
control_pre_bad = [40.0, 45.0]
placebo_did_bad = (treated_pre_bad[1]-treated_pre_bad[0]) - (control_pre_bad[1]-control_pre_bad[0])
print("placebo DiD, diverging pre-trends:        ", placebo_did_bad, " (nonzero -- assumption violated)")

## Synthetic control

#### 0. Core idea

DiD needs a comparison group whose pre-treatment trend actually resembles the treated unit's. When there's only ONE treated unit and a pool of untreated candidates, none of which is individually a great match, synthetic control builds an artificial comparison unit instead of picking one: a weighted combination of the untreated units whose pre-treatment trajectory best matches the treated unit's own pre-treatment trajectory. That synthetic unit's post-period path is then the counterfactual, what the treated unit would have done absent treatment.

It's preferred over DiD with a single hand-picked control specifically because it doesn't rely on judgment-picking one "comparable" region, the weights are fit systematically from the available untreated pool. Toy: one treated region and two untreated regions, four pre-treatment periods.

In [ ]:
import numpy as np

R1_pre = np.array([8.0, 10.0, 12.0, 14.0])
R2_pre = np.array([12.0, 14.0, 16.0, 18.0])
treated_pre = np.array([10.0, 12.0, 14.0, 16.0])

# grid-search the weight w on R1 (and 1-w on R2) that best matches the treated unit's pre-period trend
best_w, best_err = None, None
for w in np.linspace(0, 1, 1001):
    synth = w*R1_pre + (1-w)*R2_pre
    err = np.sum((synth - treated_pre)**2)
    if best_err is None or err < best_err:
        best_err, best_w = err, w
print("best weight on R1:", round(best_w, 3), " (and", round(1-best_w, 3), "on R2), squared error:", round(best_err, 6))

# post-period: one period, regions continue their trend, treated region gets a real +5 effect
R1_post, R2_post = 16.0, 20.0
synthetic_control_post = best_w*R1_post + (1-best_w)*R2_post
treated_post_actual = 18.0 + 5.0  # would-have-been 18 (continuing the trend) plus the true +5 effect

print("synthetic control counterfactual (post):", synthetic_control_post)
print("actual treated post value:               ", treated_post_actual)
print("synthetic-control treatment effect estimate:", treated_post_actual - synthetic_control_post)

#### 1. Reading the result

The weight search recovers `w=0.5` with zero pre-period error, the treated region's pre-trend is exactly the midpoint of the two untreated regions, so the optimizer finds that automatically rather than anyone guessing "average these two." Applying that same 0.5/0.5 weighting to the untreated regions' POST-period values gives a synthetic counterfactual of 18, and comparing that to the treated region's actual post value of 23 recovers the true effect of 5 exactly. With real data the match is rarely exact, the quality of a synthetic control estimate depends entirely on how well the weighted combination actually fits the PRE-period, a poor pre-period fit is a direct warning sign that the post-period estimate is not trustworthy either.

## Instrumental Variables and Regression Discontinuity (brief)

#### 0. Core idea

Both handle an UNMEASURED confounder, where PSM/IPW/DiD/synthetic control can't help since they all only adjust for confounders you actually observed or a common trend, not confounding driven by something invisible in the data. This exact situation, "confounding by indication", where a treatment is only ever given to units already flagged as different (e.g. a retention call only made to clients a model already scored as high-risk), is a common way an unmeasured confounder shows up, comparing called vs. not-called clients naively confounds the call's effect with the fact that called clients were already on a worse expected trajectory by construction.

**Instrumental variables (IV)**: find an instrument Z that affects the treatment D, but affects the outcome Y *only* through D (the "exclusion restriction"). The Wald estimator for a binary instrument: `LATE = [E(Y|Z=1) - E(Y|Z=0)] / [E(D|Z=1) - E(D|Z=0)]`. Toy example, a randomized encouragement (Z) to enroll in a program, with partial compliance (D = actual enrollment):
```
encouraged (Z=1):     enrollment rate 0.6, mean outcome 60
not encouraged (Z=0): enrollment rate 0.2, mean outcome 44
LATE = (60-44) / (0.6-0.2) = 16 / 0.4 = 40
```
This estimates the effect only for "compliers" (people whose enrollment decision the encouragement actually changed), not the whole population. For confounding-by-indication specifically, a genuine instrument (something that shifts whether the treatment happens but has no other path to the outcome, e.g. advisor call-capacity on a given day) is the cleanest fix in principle, but a real instrument satisfying the exclusion restriction is often genuinely hard to find.

**Regression discontinuity (RDD)**: when treatment is assigned by a hard cutoff on a continuous score (credit-score-based approval, a risk-score threshold that triggers a retention call), compare outcomes just below vs. just above the cutoff, since units in a tight window around the threshold are otherwise similar except for treatment assignment. This is usually the most defensible design for confounding-by-indication when the treatment is genuinely threshold-triggered. The key assumption: no manipulation of the running variable near the cutoff, if anyone (the client, the advisor, the model's own scorer) can strategically nudge a score just across the threshold to trigger or avoid treatment, the units right at the boundary are no longer comparable and RDD breaks. The naive difference of raw local means is also biased by the running variable's own slope inside the bandwidth window; a local LINEAR fit on each side, evaluated exactly at the cutoff, removes that slope bias.

In [ ]:
import numpy as np

# IV Wald estimator
ez1_y, ez0_y = 60, 44
ez1_d, ez0_d = 0.6, 0.2
wald_late = (ez1_y - ez0_y) / (ez1_d - ez0_d)
print("IV Wald / LATE estimate:", wald_late)

# RDD: simulate a true discontinuity of 8 at cutoff=50, recover it two ways
rng = np.random.default_rng(0)
score = rng.uniform(30, 70, size=2000)
true_jump = 8
Y = 0.5*score + true_jump*(score >= 50) + rng.normal(0, 1, 2000)

bw = 5
below = (score >= 50-bw) & (score < 50)
above = (score >= 50) & (score < 50+bw)

naive_jump = Y[above].mean() - Y[below].mean()
print("naive local-mean jump:", round(naive_jump, 2), " (biased upward by the slope inside the window)")

def local_linear_at_cutoff(x, y, cutoff):
    X = np.column_stack([np.ones(len(x)), x - cutoff])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef[0]  # intercept = fitted value exactly at the cutoff

left_val = local_linear_at_cutoff(score[below], Y[below], 50)
right_val = local_linear_at_cutoff(score[above], Y[above], 50)
print("local-linear jump estimate:", round(right_val - left_val, 2), " (close to the true jump of 8)")

#### 1. Why this matters beyond a textbook exercise

A backtest that just compares "outcomes for cases we happened to treat" vs. "outcomes for cases we didn't" is exactly the confounded comparison Simpson's paradox warns about, whoever gets treated is rarely a random subset. Before trusting a backtest's estimated lift, check what actually drove the treatment/no-treatment split, if it correlates with anything that also drives the outcome (case difficulty, customer tenure, risk tier), the naive comparison is biased and one of PSM/IPW/DiD/synthetic-control/IV is the more defensible route, or push for a genuine RCT (`ab-testing.ipynb`) if the stakes justify it. And when reporting any of these estimates, name the method and its assumptions explicitly rather than presenting a backtested or observational number as if it were live-validated, that honesty about what was actually measured is usually the actual point of the conversation, not the specific technique used.